# Looking at the preprocessed epochs

Run download + preprocess first:
```bash
python3 data/download.py --subjects 1-20 --runs 3-14
python3 data/preprocess.py --subjects 1-20 --runs 3-14
```

Training is in `models/csp_baseline.py` and `models/train_cnn.py`, not here.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config
from models.utils import load_epochs

ds = load_epochs()
X, y, subjects, runs = ds["X"], ds["y"], ds["subjects"], ds["runs"]
print("X", X.shape, "dtype", X.dtype)
print("subjects", np.unique(subjects).tolist())
print("class counts", {n: int(np.sum(y == i)) for i, n in enumerate(config.CLASS_NAMES)})
print("majority", float(np.max(np.bincount(y)) / len(y)))

Feet only show up as T2 on the fists-vs-feet runs, so there are fewer of them. I look at macro-F1 for that reason.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = [np.sum(y == i) for i in range(len(config.CLASS_NAMES))]
axes[0].bar(config.CLASS_NAMES, counts, color="#2E86AB")
axes[0].set_ylabel("Epochs")
axes[0].set_title("Class counts")
axes[0].tick_params(axis="x", rotation=25)

per = [np.sum(subjects == s) for s in np.unique(subjects)]
axes[1].hist(per, bins=8, color="#E94F37", edgecolor="white")
axes[1].set_xlabel("Epochs per subject")
axes[1].set_title("After dropping extra rest trials")
fig.tight_layout()
plt.show()

Average traces over all channels. This is just a check that the data loaded (finite, reasonable uV). MI is mostly spatial covariance, not a big ERP.

In [ ]:
times = np.linspace(config.EPOCH_TMIN, config.EPOCH_TMAX, X.shape[-1])
fig, ax = plt.subplots(figsize=(8, 4))
for i, name in enumerate(config.CLASS_NAMES):
    grand = X[y == i].mean(axis=(0, 1))
    ax.plot(times, grand, label=name, lw=1.4)
ax.set_xlabel("Time (s)")
ax.set_ylabel("uV (mean over trials and channels)")
ax.set_title("Grand average")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()
print("finite", np.isfinite(X).all(), "min/max", float(X.min()), float(X.max()))

Don't shuffle all epochs together for train/test. That leaks the same person into both sides. The training scripts split per subject or leave a whole subject out.

In [ ]:
print("epochs per subject:")
for s in np.unique(subjects):
    mask = subjects == s
    hist = np.bincount(y[mask], minlength=len(config.CLASS_NAMES))
    print(f"  S{int(s):03d}  n={int(mask.sum()):3d}  {hist.tolist()}")